In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

public enum OrderStatus
{
    Created,
    InProgress,
    Completed
}

public delegate void OrderStatusChangedHandler(Order order, OrderStatus oldStatus, OrderStatus newStatus);

public class Customer
{
    public string FirstName { get; set; }
    public string LastName { get; set; }
    public string PhoneNumber { get; set; }

    public Customer(string firstName, string lastName, string phoneNumber)
    {
        FirstName = firstName;
        LastName = lastName;
        PhoneNumber = phoneNumber;
    }

    public string GetFullName()
    {
        return $"{FirstName} {LastName}";
    }

    public void UpdatePhoneNumber(string newPhone)
    {
        PhoneNumber = newPhone;
    }

    public void DisplayInfo()
    {
        Console.WriteLine($"Заказчик: {GetFullName()} | Телефон: {PhoneNumber}");
    }
}

public class Employee
{
    public string FirstName { get; set; }
    public string LastName { get; set; }
    public string Position { get; set; }
    public int OrdersProcessed { get; private set; }

    public Employee(string firstName, string lastName, string position)
    {
        FirstName = firstName;
        LastName = lastName;
        Position = position;
        OrdersProcessed = 0;
    }

    public string GetFullName()
    {
        return $"{FirstName} {LastName}";
    }

    public void IncrementProcessedOrders()
    {
        OrdersProcessed++;
    }

    public void ReceiveNotification(string message)
    {
        Console.WriteLine($"  [Уведомление для {GetFullName()} ({Position})]: {message}");
    }

    public void DisplayInfo()
    {
        Console.WriteLine($"Сотрудник: {GetFullName()} | Должность: {Position} | Обработано заказов: {OrdersProcessed}");
    }
}

public class Order
{
    public int OrderId { get; set; }
    public string Description { get; set; }
    public Customer Customer { get; set; }
    public Employee AssignedEmployee { get; set; }
    public OrderStatus Status { get; private set; }
    public DateTime CreatedAt { get; set; }

    public Order(int orderId, string description, Customer customer)
    {
        OrderId = orderId;
        Description = description;
        Customer = customer;
        Status = OrderStatus.Created;
        CreatedAt = DateTime.Now;
    }

    public void AssignEmployee(Employee employee)
    {
        AssignedEmployee = employee;
    }

    public void SetStatus(OrderStatus newStatus)
    {
        Status = newStatus;
    }

    public void DisplayInfo()
    {
        string employeeName = AssignedEmployee != null ? AssignedEmployee.GetFullName() : "Не назначен";
        Console.WriteLine($"Заказ #{OrderId} [{Status}] от {CreatedAt:dd.MM.yyyy HH:mm}");
        Console.WriteLine($"  Описание: {Description}");
        Console.WriteLine($"  Клиент: {Customer.GetFullName()} ({Customer.PhoneNumber})");
        Console.WriteLine($"  Ответственный: {employeeName}");
    }
}

public class Company
{
    public string Name { get; set; }
    public List<Employee> Employees { get; private set; } = new List<Employee>();
    public List<Order> Orders { get; private set; } = new List<Order>();
    public List<Customer> Customers { get; private set; } = new List<Customer>();

    private int _nextOrderId = 1;

    public event OrderStatusChangedHandler OnOrderStatusChanged;

    public Company(string name)
    {
        Name = name;
    }

    public void AddEmployee(Employee employee)
    {
        if (employee != null && !Employees.Contains(employee))
        {
            Employees.Add(employee);
        }
    }

    public void AddCustomer(Customer customer)
    {
        if (customer != null && !Customers.Contains(customer))
        {
            Customers.Add(customer);
        }
    }

    public Order CreateOrder(string description, Customer customer)
    {
        AddCustomer(customer);
        Order order = new Order(_nextOrderId++, description, customer);
        Orders.Add(order);
        Console.WriteLine($"Создан заказ #{order.OrderId}: '{order.Description}' для клиента {customer.GetFullName()}");
        return order;
    }

    public void AssignEmployeeToOrder(int orderId, Employee employee)
    {
        Order order = Orders.FirstOrDefault(o => o.OrderId == orderId);
        if (order == null)
        {
            Console.WriteLine($"Заказ #{orderId} не найден.");
            return;
        }

        order.AssignEmployee(employee);
        Console.WriteLine($"Сотрудник {employee.GetFullName()} назначен ответственным за заказ #{orderId}.");
    }

    public void ChangeOrderStatus(int orderId, OrderStatus newStatus)
    {
        Order order = Orders.FirstOrDefault(o => o.OrderId == orderId);
        if (order == null)
        {
            Console.WriteLine($"Заказ #{orderId} не найден.");
            return;
        }

        OrderStatus oldStatus = order.Status;
        if (oldStatus == newStatus) return;

        order.SetStatus(newStatus);

        if (newStatus == OrderStatus.Completed && order.AssignedEmployee != null)
        {
            order.AssignedEmployee.IncrementProcessedOrders();
        }

        OnOrderStatusChanged?.Invoke(order, oldStatus, newStatus);
    }

    public List<Order> GetOrdersByStatus(OrderStatus status)
    {
        return Orders.Where(o => o.Status == status).ToList();
    }

    public List<Order> FindOrdersByCustomer(string lastName)
    {
        return Orders.Where(o => o.Customer.LastName.Equals(lastName, StringComparison.OrdinalIgnoreCase)).ToList();
    }

    public void PrintEmployeeReport()
    {
        Console.WriteLine($"=== Отчет по сотрудникам компании '{Name}' ===");
        foreach (var emp in Employees)
        {
            emp.DisplayInfo();
        }
    }
}

Company company = new Company("Интертеймент Солюшнс");

Employee emp1 = new Employee("Алексей", "Смирнов", "Старший специалист");
Employee emp2 = new Employee("Мария", "Кузнецова", "Менеджер проектов");
company.AddEmployee(emp1);
company.AddEmployee(emp2);

company.OnOrderStatusChanged += (order, oldStatus, newStatus) =>
{
    Console.WriteLine($"[СОБЫТИЕ] Статус заказа #{order.OrderId} изменен: {oldStatus} -> {newStatus}");
    if (order.AssignedEmployee != null)
    {
        order.AssignedEmployee.ReceiveNotification($"Статус вашего заказа #{order.OrderId} ('{order.Description}') изменился на '{newStatus}'");
    }
};

Customer cust1 = new Customer("Иван", "Сергеев", "+7 (912) 345-67-89");
Customer cust2 = new Customer("Ольга", "Иванова", "+7 (922) 987-65-43");

Console.WriteLine("=== Создание заказов ===");
Order order1 = company.CreateOrder("Разработка корпоративного сайта", cust1);
Order order2 = company.CreateOrder("Настройка CRM системы", cust2);
Order order3 = company.CreateOrder("SEO оптимизация и аудит", cust1);

Console.WriteLine("\n=== Назначение ответственных ===");
company.AssignEmployeeToOrder(order1.OrderId, emp1);
company.AssignEmployeeToOrder(order2.OrderId, emp2);
company.AssignEmployeeToOrder(order3.OrderId, emp1);

Console.WriteLine("\n=== Изменение статусов заказов ===");
company.ChangeOrderStatus(order1.OrderId, OrderStatus.InProgress);
company.ChangeOrderStatus(order1.OrderId, OrderStatus.Completed);
company.ChangeOrderStatus(order2.OrderId, OrderStatus.InProgress);

Console.WriteLine("\n=== Заказы в статусе 'InProgress' ===");
var inProgressOrders = company.GetOrdersByStatus(OrderStatus.InProgress);
foreach (var o in inProgressOrders)
{
    o.DisplayInfo();
}

Console.WriteLine("\n=== Поиск заказов по фамилии заказчика 'Сергеев' ===");
var custOrders = company.FindOrdersByCustomer("Сергеев");
foreach (var o in custOrders)
{
    o.DisplayInfo();
}

Console.WriteLine();
company.PrintEmployeeReport();
